# K-FOLD CROSS-VALIDATION CHO TabNet + SMOTE (mở rộng mục 3.4.5 / 4.3.1)

Notebook này giải quyết đúng hạn chế đã nêu ở mục 5.2: kết quả "TabNet + SMOTE" hiện tại
(Bảng 4.2, mục 4.3.1) chỉ được đánh giá trên **một lần chia** train/val/test duy nhất, trong
khi K-fold của TabNet gốc (mục 3.4.5/4.3.2) đã cho thấy Macro F1 dao động khá mạnh giữa các
fold — đặc biệt ở các lớp thiểu số mà SMOTE tác động (floodattack, mitmattack). Notebook này
chạy **Stratified 5-fold Cross-Validation cho biến thể TabNet + SMOTE**, để có thể so sánh
mean ± std với kết quả K-fold của TabNet gốc một cách công bằng.

### Nguyên tắc bắt buộc — tránh rò rỉ dữ liệu (data leakage)
SMOTE chỉ được fit **bên trong từng fold, chỉ trên phần train của fold đó**. Phần test của
fold giữ nguyên phân bố gốc, không SMOTE, không hề "nhìn thấy" trước khi đánh giá. Đây là lý
do bắt buộc phải tự viết vòng lặp K-fold thủ công thay vì dùng `cross_val_score` — thứ tự
đúng luôn phải là: **chia fold → SMOTE trên train fold → huấn luyện → đánh giá trên test fold
chưa đụng tới**, không bao giờ được làm ngược lại.

### Nguồn dữ liệu
Notebook gộp lại `X_train.csv` + `X_val.csv` + `X_test.csv` (đã tiền xử lý/chuẩn hóa sẵn từ
`pre.ipynb`, dùng chung cho `TABNET.ipynb`) thành một pool đầy đủ, sau đó chia lại bằng
`StratifiedKFold` — đúng cách mà K-fold của TabNet gốc (mục 3.4.5) đã làm, để hai bảng kết
quả (gốc vs SMOTE) dùng chung một cách chia fold và có thể so sánh trực tiếp.

**⚠ Lưu ý quan trọng:** nếu notebook K-fold TabNet gốc của bạn dùng cấu hình `max_epochs` /
`patience` giảm để kịp thời gian (mục 3.4.5 có nhắc điều này), hãy chỉnh `MAX_EPOCHS_CV` và
`PATIENCE_CV` ở Cell cấu hình bên dưới cho **khớp đúng số** đã dùng ở đó — nếu hai bên dùng
cấu hình khác nhau, phần so sánh mean ± std sẽ không còn công bằng.

**Output:** `results_kfold_smote.json` (per-fold + mean±std, cùng schema hướng dùng cho
dashboard) và `kfold_smote_macro_f1_boxplot.png`.


In [1]:
!pip install pytorch-tabnet imbalanced-learn


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import time
import json
import pickle
import torch
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import label_binarize
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    confusion_matrix,
)
from imblearn.over_sampling import SMOTE
from pytorch_tabnet.tab_model import TabNetClassifier

import matplotlib.pyplot as plt
import seaborn as sns


In [3]:
# ==============================================================================
# 1. CẤU HÌNH — khớp với pre_smote.ipynb / train_tabnet_smote.ipynb / TABNET.ipynb
# ==============================================================================
data_dir = "../data/processed/"

RANDOM_STATE = 42          # seed dùng xuyên suốt: KFold, SMOTE, TabNet, inner val split
N_SPLITS = 5                # số fold, khớp K=5 dùng cho K-fold TabNet gốc (mục 3.4.5)

# --- Cấu hình SMOTE, khớp pre_smote.ipynb ---
TARGET_COUNT = 500              # ngưỡng nâng lên, đúng Bảng 4.2
MIN_CLASS_SIZE_TO_TOUCH = 500   # chỉ SMOTE các lớp có số mẫu train-fold < ngưỡng này
DEFAULT_K_NEIGHBORS = 5         # k_neighbors mặc định của SMOTE

# --- Cấu hình huấn luyện TabNet cho K-fold ---
# ✓ Đã đối chiếu với kfold_tabnet.py (K-fold TabNet gốc, mục 3.4.5): MAX_EPOCHS=40,
# PATIENCE=8 — khớp đúng, giữ nguyên để hai bảng kết quả (gốc vs SMOTE) so sánh công bằng.
MAX_EPOCHS_CV = 40
PATIENCE_CV = 8

# Tỉ lệ tách validation nội bộ (chỉ để early-stopping) từ phần train của mỗi fold,
# sau khi đã SMOTE — KHÔNG lấy từ phần test của fold để tránh rò rỉ thông tin đánh giá.
INNER_VAL_RATIO = 0.10

# Tham số kiến trúc TabNet — GIỮ NGUYÊN như TABNET.ipynb / train_tabnet_smote.ipynb
tabnet_params = {
    "n_d": 64,
    "n_a": 64,
    "n_steps": 5,
    "gamma": 1.5,
    "n_independent": 2,
    "n_shared": 2,
    "lambda_sparse": 1e-4,
    "momentum": 0.3,
    "clip_value": 2.0,
    "optimizer_fn": torch.optim.Adam,
    "optimizer_params": dict(lr=2e-2),
    "scheduler_fn": torch.optim.lr_scheduler.StepLR,
    "scheduler_params": {"step_size": 10, "gamma": 0.9},
    "mask_type": "entmax",
    "verbose": 0,          # tắt log epoch-by-epoch để output không quá dài qua 5 fold
    "seed": RANDOM_STATE,
}

print(f"N_SPLITS={N_SPLITS} | MAX_EPOCHS_CV={MAX_EPOCHS_CV} | PATIENCE_CV={PATIENCE_CV} "
      f"| TARGET_COUNT={TARGET_COUNT}")


N_SPLITS=5 | MAX_EPOCHS_CV=40 | PATIENCE_CV=8 | TARGET_COUNT=500


In [4]:
# ==============================================================================
# 2. TẢI DỮ LIỆU VÀ GỘP LẠI THÀNH MỘT POOL ĐẦY ĐỦ
#    (đúng cách K-fold TabNet gốc mục 3.4.5 đã làm, để dùng chung một cách chia fold)
# ==============================================================================
X_train_o = pd.read_csv(data_dir + "X_train.csv")
X_val_o   = pd.read_csv(data_dir + "X_val.csv")
X_test_o  = pd.read_csv(data_dir + "X_test.csv")

y_train_o = pd.read_csv(data_dir + "y_train.csv").values.ravel()
y_val_o   = pd.read_csv(data_dir + "y_val.csv").values.ravel()
y_test_o  = pd.read_csv(data_dir + "y_test.csv").values.ravel()

with open(data_dir + "label_encoder.pkl", "rb") as f:
    label_encoder = pickle.load(f)
label_names = label_encoder.classes_
feature_names = X_train_o.columns.tolist()

X_full = pd.concat([X_train_o, X_val_o, X_test_o], axis=0, ignore_index=True)
y_full = np.concatenate([y_train_o, y_val_o, y_test_o])

print(f"Pool đầy đủ: X_full={X_full.shape}, y_full={y_full.shape}")
print(f"Số lớp: {len(label_names)}, số features: {len(feature_names)}")

# Lưu ý phương pháp luận: X_full đã được chuẩn hóa (z-score) MỘT LẦN bằng scaler fit trên
# tập train của lần chia 70/15/15 gốc (mục 3.3.5), không refit lại cho từng fold ở đây.
# Đây là một đơn giản hóa hợp lý miễn là K-fold TabNet gốc cũng làm giống vậy (khớp mục
# 3.4.5); nếu muốn chặt chẽ tuyệt đối, cần format lại pipeline để refit StandardScaler
# bên trong mỗi fold. Nên nêu rõ điểm này trong mục 5.2 (Hạn chế) nếu không refit lại.


Pool đầy đủ: X_full=(319971, 78), y_full=(319971,)
Số lớp: 8, số features: 78


In [5]:
# ==============================================================================
# 3. HÀM ÁP DỤNG SMOTE — TÁI SỬ DỤNG ĐÚNG LOGIC CỦA pre_smote.ipynb
#    (chỉ khác: nhận X, y của MỘT FOLD TRAIN thay vì toàn bộ X_train gốc)
# ==============================================================================
def apply_smote_fold(X_fold_train, y_fold_train, fold_idx):
    """Áp dụng đúng quy tắc SMOTE của mục 3.4.4/pre_smote.ipynb trên train của 1 fold.
    Trả về X, y đã SMOTE và dict thống kê trước/sau để log lại.
    """
    from collections import Counter
    counts_before = Counter(y_fold_train)

    minority_classes = [idx for idx in range(len(label_names))
                         if counts_before.get(idx, 0) < MIN_CLASS_SIZE_TO_TOUCH]

    X_res, y_res = X_fold_train.values.copy(), y_fold_train.copy()

    for cls_idx in minority_classes:
        n_current = int(np.sum(y_res == cls_idx))
        if n_current >= TARGET_COUNT or n_current < 2:
            continue

        # Công thức k_neighbors ĐÚNG NHƯ pre_smote.ipynb — không phải lúc nào cũng hạ,
        # chỉ hạ khi lớp có rất ít mẫu (n_current <= DEFAULT_K_NEIGHBORS + 1).
        k = min(DEFAULT_K_NEIGHBORS, max(1, n_current - 1))

        smote = SMOTE(
            sampling_strategy={cls_idx: TARGET_COUNT},
            k_neighbors=k,
            random_state=RANDOM_STATE + fold_idx,   # seed lệch theo fold để không lặp y hệt
        )
        X_res, y_res = smote.fit_resample(X_res, y_res)

    counts_after = Counter(y_res)
    stats = {
        "before": {label_names[i]: int(counts_before.get(i, 0)) for i in range(len(label_names))},
        "after":  {label_names[i]: int(counts_after.get(i, 0)) for i in range(len(label_names))},
        "smote_classes": [label_names[i] for i in minority_classes],
    }
    X_res_df = pd.DataFrame(X_res, columns=feature_names)
    return X_res_df, y_res, stats


In [6]:
# ==============================================================================
# 4. VÒNG LẶP K-FOLD CHÍNH
#    Với mỗi fold: chia fold -> SMOTE trên train fold -> tách val nội bộ (early-stopping)
#    -> huấn luyện TabNet từ đầu -> đánh giá trên test fold GỐC (không SMOTE)
# ==============================================================================
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

fold_results = []
fold_smote_stats = []

X_full_values = X_full.values
t_total0 = time.perf_counter()

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(X_full_values, y_full), start=1):
    print(f"\n{'='*70}\nFOLD {fold_idx}/{N_SPLITS}\n{'='*70}")
    t_fold0 = time.perf_counter()

    X_fold_train_raw = X_full.iloc[train_idx].reset_index(drop=True)
    y_fold_train_raw = y_full[train_idx]
    X_fold_test = X_full.iloc[test_idx].reset_index(drop=True)
    y_fold_test = y_full[test_idx]

    # --- 4.1 SMOTE chỉ trên train fold ---
    X_fold_train_sm, y_fold_train_sm, smote_stats = apply_smote_fold(
        X_fold_train_raw, y_fold_train_raw, fold_idx
    )
    fold_smote_stats.append(smote_stats)
    print(f"  Train fold: {X_fold_train_raw.shape[0]:,d} -> sau SMOTE: "
          f"{X_fold_train_sm.shape[0]:,d} mẫu | Test fold (gốc, không SMOTE): "
          f"{X_fold_test.shape[0]:,d} mẫu")
    print(f"  Lớp được SMOTE ở fold này: {smote_stats['smote_classes']}")

    # --- 4.2 Tách validation nội bộ để early-stopping (KHÔNG dùng test fold) ---
    X_tr, X_inner_val, y_tr, y_inner_val = train_test_split(
        X_fold_train_sm, y_fold_train_sm,
        test_size=INNER_VAL_RATIO,
        stratify=y_fold_train_sm,
        random_state=RANDOM_STATE,
    )

    # --- 4.3 Huấn luyện TabNet từ đầu (cấu hình kiến trúc giữ nguyên) ---
    model = TabNetClassifier(**tabnet_params)
    model.fit(
        X_train=X_tr.values,
        y_train=y_tr,
        eval_set=[(X_tr.values, y_tr), (X_inner_val.values, y_inner_val)],
        eval_name=["train", "inner_valid"],
        eval_metric=["accuracy", "balanced_accuracy"],
        max_epochs=MAX_EPOCHS_CV,
        patience=PATIENCE_CV,
        batch_size=1024,
        virtual_batch_size=128,
        num_workers=0,
        drop_last=False,
    )

    # --- 4.4 Đánh giá trên test fold GỐC (chưa từng SMOTE, chưa từng thấy khi train) ---
    y_pred = model.predict(X_fold_test.values)
    y_pred_proba = model.predict_proba(X_fold_test.values)

    acc = accuracy_score(y_fold_test, y_pred)
    precision, recall, f1, support = precision_recall_fscore_support(
        y_fold_test, y_pred, average=None, labels=range(len(label_names)), zero_division=0
    )
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(
        y_fold_test, y_pred, average="macro", zero_division=0
    )
    _, _, weighted_f1, _ = precision_recall_fscore_support(
        y_fold_test, y_pred, average="weighted", zero_division=0
    )

    try:
        y_test_bin = label_binarize(y_fold_test, classes=range(len(label_names)))
        auc_scores = []
        for i in range(len(label_names)):
            try:
                auc_scores.append(roc_auc_score(y_test_bin[:, i], y_pred_proba[:, i]))
            except Exception:
                auc_scores.append(np.nan)
        macro_auc = float(np.nanmean(auc_scores))
    except Exception:
        macro_auc = np.nan

    fold_time = time.perf_counter() - t_fold0
    n_epochs_actual = len(model.history.history["loss"])

    result = {
        "fold": fold_idx,
        "accuracy": float(acc),
        "macro_precision": float(macro_p),
        "macro_recall": float(macro_r),
        "macro_f1": float(macro_f1),
        "weighted_f1": float(weighted_f1),
        "macro_auc": macro_auc,
        "n_epochs_actual": n_epochs_actual,
        "fold_time_sec": fold_time,
        "per_class_f1": {label_names[i]: float(f1[i]) for i in range(len(label_names))},
        "test_support": {label_names[i]: int(support[i]) for i in range(len(label_names))},
    }
    fold_results.append(result)

    print(f"  -> Accuracy={acc:.4f} | Macro F1={macro_f1:.4f} | Weighted F1={weighted_f1:.4f} "
          f"| Macro AUC={macro_auc:.4f} | {n_epochs_actual} epochs | {fold_time:.1f}s")

total_time = time.perf_counter() - t_total0
print(f"\n✓ Hoàn tất {N_SPLITS} fold trong {total_time/60:.1f} phút.")



FOLD 1/5
  Train fold: 255,976 -> sau SMOTE: 257,165 mẫu | Test fold (gốc, không SMOTE): 63,995 mẫu
  Lớp được SMOTE ở fold này: ['attackfree', 'floodattack', 'mitmattack']

Early stopping occurred at epoch 15 with best_epoch = 7 and best_inner_valid_balanced_accuracy = 0.97561
  -> Accuracy=0.8980 | Macro F1=0.8204 | Weighted F1=0.9270 | Macro AUC=0.9990 | 16 epochs | 4133.9s

FOLD 2/5
  Train fold: 255,977 -> sau SMOTE: 257,165 mẫu | Test fold (gốc, không SMOTE): 63,994 mẫu
  Lớp được SMOTE ở fold này: ['attackfree', 'floodattack', 'mitmattack']

Early stopping occurred at epoch 10 with best_epoch = 2 and best_inner_valid_balanced_accuracy = 0.96384
  -> Accuracy=0.9940 | Macro F1=0.8988 | Weighted F1=0.9941 | Macro AUC=0.9980 | 11 epochs | 2639.0s

FOLD 3/5
  Train fold: 255,977 -> sau SMOTE: 257,166 mẫu | Test fold (gốc, không SMOTE): 63,994 mẫu
  Lớp được SMOTE ở fold này: ['attackfree', 'floodattack', 'mitmattack']

Early stopping occurred at epoch 25 with best_epoch = 17 and be

In [7]:
# ==============================================================================
# 5. TỔNG HỢP MEAN ± STD QUA CÁC FOLD
# ==============================================================================
metrics_keys = ["accuracy", "macro_precision", "macro_recall", "macro_f1",
                 "weighted_f1", "macro_auc"]

summary = {}
print(f"{'Metric':<20} {'Mean':>10} {'Std':>10}")
print("-" * 42)
for key in metrics_keys:
    values = np.array([r[key] for r in fold_results], dtype=float)
    mean_v, std_v = float(np.nanmean(values)), float(np.nanstd(values))
    summary[key] = {"mean": mean_v, "std": std_v, "per_fold": values.tolist()}
    print(f"{key:<20} {mean_v:>10.4f} {std_v:>10.4f}")

print("\nSo sánh nhanh với khoảng dao động K-fold TabNet gốc đã có trong luận văn "
      "(mục 4.3.2, Macro F1: 0.8261–0.9375) — điền tay nếu cần đối chiếu trực quan hơn.")


Metric                     Mean        Std
------------------------------------------
accuracy                 0.9742     0.0382
macro_precision          0.8709     0.0351
macro_recall             0.9582     0.0203
macro_f1                 0.8928     0.0377
weighted_f1              0.9802     0.0266
macro_auc                0.9991     0.0006

So sánh nhanh với khoảng dao động K-fold TabNet gốc đã có trong luận văn (mục 4.3.2, Macro F1: 0.8261–0.9375) — điền tay nếu cần đối chiếu trực quan hơn.


In [8]:
# ==============================================================================
# 6. LƯU KẾT QUẢ
# ==============================================================================
output = {
    "config": {
        "n_splits": N_SPLITS,
        "random_state": RANDOM_STATE,
        "target_count": TARGET_COUNT,
        "min_class_size_to_touch": MIN_CLASS_SIZE_TO_TOUCH,
        "max_epochs_cv": MAX_EPOCHS_CV,
        "patience_cv": PATIENCE_CV,
        "inner_val_ratio": INNER_VAL_RATIO,
    },
    "fold_results": fold_results,
    "fold_smote_stats": fold_smote_stats,
    "summary": summary,
}

with open(data_dir + "results_kfold_smote.json", "w", encoding="utf-8") as f:
    json.dump(output, f, indent=4, ensure_ascii=False)

print(f"✓ Đã lưu: {data_dir}results_kfold_smote.json")


✓ Đã lưu: ../data/processed/results_kfold_smote.json


In [9]:
# ==============================================================================
# 7. BIỂU ĐỒ BOXPLOT MACRO F1 QUA CÁC FOLD — SO SÁNH VỚI TabNet GỐC
#    File K-fold TabNet gốc: đúng như kfold_tabnet.py đã lưu ra
#    ("kfold_tabnet_results.json" — file tổng hợp cuối cùng, có đủ
#    fold_results + summary; KHÔNG dùng "kfold_tabnet_results_partial.json"
#    vì đó chỉ là bản lưu tạm sau mỗi fold, phòng bị ngắt giữa chừng).
# ==============================================================================
macro_f1_smote = [r["macro_f1"] for r in fold_results]

data_to_plot = [macro_f1_smote]
labels_to_plot = ["TabNet + SMOTE\n(5-fold)"]

BASE_KFOLD_FILENAME = "kfold_tabnet_results.json"  # đúng tên do kfold_tabnet.py lưu ra

try:
    with open(data_dir + BASE_KFOLD_FILENAME, "r", encoding="utf-8") as f:
        base_kfold = json.load(f)
    base_macro_f1 = [r["macro_f1"] for r in base_kfold.get("fold_results", [])]
    if base_macro_f1:
        data_to_plot.insert(0, base_macro_f1)
        labels_to_plot.insert(0, "TabNet gốc\n(5-fold)")
        print(f"  Đã tìm thấy và nạp kết quả K-fold TabNet gốc từ: {BASE_KFOLD_FILENAME}")
except FileNotFoundError:
    print(f"  (Không tìm thấy {BASE_KFOLD_FILENAME} — chỉ vẽ riêng kết quả SMOTE. "
          f"Hãy chạy kfold_tabnet.py trước để tạo file này.)")

  Đã tìm thấy và nạp kết quả K-fold TabNet gốc từ: kfold_tabnet_results.json


In [10]:
# ==============================================================================
# 8. TÓM TẮT CUỐI CÙNG
# ==============================================================================
print("=" * 70)
print("  KẾT QUẢ K-FOLD (5-fold) — TabNet + SMOTE")
print("=" * 70)
for r in fold_results:
    print(f"  Fold {r['fold']}: Macro F1={r['macro_f1']:.4f} | "
          f"Accuracy={r['accuracy']:.4f} | Weighted F1={r['weighted_f1']:.4f} | "
          f"{r['n_epochs_actual']} epochs")
print("-" * 70)
print(f"  Macro F1  : {summary['macro_f1']['mean']:.4f} ± {summary['macro_f1']['std']:.4f}")
print(f"  Accuracy  : {summary['accuracy']['mean']:.4f} ± {summary['accuracy']['std']:.4f}")
print(f"  Weighted F1: {summary['weighted_f1']['mean']:.4f} ± {summary['weighted_f1']['std']:.4f}")
print(f"  Macro AUC : {summary['macro_auc']['mean']:.4f} ± {summary['macro_auc']['std']:.4f}")
print("=" * 70)
print("\nSo sánh mean ± std này với K-fold TabNet gốc (mục 4.3.2) để trả lời câu hỏi: "
      "mức cải thiện +1.11 điểm Macro F1 quan sát được ở lần chia đơn (mục 4.3.1) có "
      "ổn định qua nhiều cách chia dữ liệu khác nhau hay không, hay chỉ là may mắn của "
      "một lần chia cụ thể.")


  KẾT QUẢ K-FOLD (5-fold) — TabNet + SMOTE
  Fold 1: Macro F1=0.8204 | Accuracy=0.8980 | Weighted F1=0.9270 | 16 epochs
  Fold 2: Macro F1=0.8988 | Accuracy=0.9940 | Weighted F1=0.9941 | 11 epochs
  Fold 3: Macro F1=0.9288 | Accuracy=0.9937 | Weighted F1=0.9939 | 26 epochs
  Fold 4: Macro F1=0.9132 | Accuracy=0.9961 | Weighted F1=0.9961 | 22 epochs
  Fold 5: Macro F1=0.9029 | Accuracy=0.9891 | Weighted F1=0.9898 | 16 epochs
----------------------------------------------------------------------
  Macro F1  : 0.8928 ± 0.0377
  Accuracy  : 0.9742 ± 0.0382
  Weighted F1: 0.9802 ± 0.0266
  Macro AUC : 0.9991 ± 0.0006

So sánh mean ± std này với K-fold TabNet gốc (mục 4.3.2) để trả lời câu hỏi: mức cải thiện +1.11 điểm Macro F1 quan sát được ở lần chia đơn (mục 4.3.1) có ổn định qua nhiều cách chia dữ liệu khác nhau hay không, hay chỉ là may mắn của một lần chia cụ thể.
